[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/01_matrix_calculus_backprop/01_matrix_calculus_backprop.ipynb)

# 01 · 矩阵微积分与反向传播

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 在真实的 **UCI 乳腺癌（WDBC）** 数据上，从零手写前向、反向传播、数值梯度校验，并训练一个两层网络。

**你将完成：**
1. sigmoid / BCE 的前向与那个著名的 `∂L/∂z = p - y`
2. 线性层反向的"上游 × 局部"模式（`dW, db, dX`）
3. **数值梯度校验**——手写 backprop 的唯一安全网
4. 在真实数据上训练两层 MLP，打败 majority baseline

> 数据：569 个肿瘤样本、30 个特征、二分类（恶性 M / 良性 B）。

## 0 · 加载真实数据（UCI 乳腺癌 WDBC）

In [ ]:
import os, urllib.request
import numpy as np
np.set_printoptions(precision=4, suppress=True)

CACHE = os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE, exist_ok=True)
def fetch(url, fname):
    p = os.path.join(CACHE, fname)
    if not os.path.exists(p): urllib.request.urlretrieve(url, p)
    return p

import pandas as pd
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
df = pd.read_csv(fetch(URL, "wdbc.data"), header=None)
# 列0=ID, 列1=诊断(M/B), 列2..31 = 30个特征
y_all = (df[1].to_numpy() == "M").astype(float)   # 恶性=1
X_all = df.iloc[:, 2:].to_numpy(dtype=float)
print("X:", X_all.shape, " 恶性占比:", y_all.mean().round(3))

# 分层划分 + 标准化（只在 train 上 fit）
rng = np.random.default_rng(0)
idx = rng.permutation(len(y_all)); cut = int(0.7*len(idx))
tr, te = idx[:cut], idx[cut:]
mu, sd = X_all[tr].mean(0), X_all[tr].std(0)
Xtr, Xte = (X_all[tr]-mu)/sd, (X_all[te]-mu)/sd
ytr, yte = y_all[tr], y_all[te]
baseline = max(yte.mean(), 1-yte.mean())
print(f"train={len(tr)} test={len(te)}  majority baseline acc={baseline:.3f}")

## 1 · sigmoid + BCE 与 `∂L/∂z = p - y`

logistic 回归：$z=Xw+b$，$p=\sigma(z)$，loss 是平均 BCE。
课上推过：loss 对 logit $z$ 的梯度恰好是 $(p-y)/n$。下面**数值验证**这个漂亮结论。

In [ ]:
def sigmoid(z): return 1.0/(1.0+np.exp(-z))

def bce(p, y):
    eps = 1e-12
    return -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))

# 取一小批，构造任意 z，验证 dL/dz = (p-y)/n
z = rng.normal(size=8); yb = (rng.random(8) > 0.5).astype(float)
p = sigmoid(z)
analytic = (p - yb)/len(yb)
# 数值梯度
eps = 1e-6; numeric = np.zeros_like(z)
for i in range(len(z)):
    zp = z.copy(); zp[i]+=eps; zm = z.copy(); zm[i]-=eps
    numeric[i] = (bce(sigmoid(zp), yb) - bce(sigmoid(zm), yb))/(2*eps)
print("解析 dL/dz:", analytic)
print("数值 dL/dz:", numeric)
print("最大误差:", np.abs(analytic-numeric).max(), " => ∂L/∂z = (p-y)/n 成立 ✓")

## 2 · 两层 MLP 的前向 + 手写反向

结构：$X \to (W_1,b_1) \to \text{ReLU} \to (W_2,b_2) \to \text{sigmoid} \to \text{BCE}$。
反向全用"上游 × 局部"模式。注意每个梯度的形状必须和被求导的参数一致。

In [ ]:
def init(din, dh, seed=0):
    g = np.random.default_rng(seed)
    return dict(
        W1=g.normal(0, np.sqrt(2/din), (din, dh)), b1=np.zeros(dh),
        W2=g.normal(0, np.sqrt(2/dh), (dh, 1)),    b2=np.zeros(1))

def forward(P, X):
    z1 = X @ P["W1"] + P["b1"]; a1 = np.maximum(0, z1)        # ReLU
    z2 = (a1 @ P["W2"] + P["b2"]).ravel(); p = sigmoid(z2)
    cache = (X, z1, a1, z2, p)
    return p, cache

def backward(P, cache, y):
    X, z1, a1, z2, p = cache; n = len(y)
    dz2 = (p - y)/n                                  # (n,)  ∂L/∂z2
    dW2 = a1.T @ dz2[:, None]                         # (dh,1)
    db2 = dz2.sum(keepdims=True)
    da1 = dz2[:, None] @ P["W2"].T                    # (n,dh) 上游
    dz1 = da1 * (z1 > 0)                              # ReLU 局部导数
    dW1 = X.T @ dz1                                   # (din,dh)
    db1 = dz1.sum(0)
    return dict(W1=dW1, b1=db1, W2=dW2, b2=db2)

P = init(Xtr.shape[1], 16)
p, cache = forward(P, Xtr)
grads = backward(P, cache, ytr)
print("初始 train loss:", bce(p, ytr).round(4))
for k in P: print(f"  grad {k}: shape {grads[k].shape}  == param {P[k].shape}? {grads[k].shape==P[k].shape}")

## 3 · 数值梯度校验整张网络

只验证一个参数（`W2` 的几个元素）即可——若它对，"上游×局部"链路就大概率全对。
中心差分，相对误差 `< 1e-5` 视为通过。

In [ ]:
def loss_of(P):
    p, _ = forward(P, Xtr); return bce(p, ytr)

def numgrad_param(P, name, k=5):
    base = P[name].ravel().copy(); ng = np.zeros(k); eps = 1e-5
    for i in range(k):
        Pp = {kk: vv.copy() for kk, vv in P.items()}
        Pm = {kk: vv.copy() for kk, vv in P.items()}
        Pp[name].ravel()[i] += eps; Pm[name].ravel()[i] -= eps
        ng[i] = (loss_of(Pp) - loss_of(Pm))/(2*eps)
    return ng

ana = backward(P, forward(P, Xtr)[1], ytr)["W2"].ravel()[:5]
num = numgrad_param(P, "W2", 5)
rel = np.abs(ana-num)/(np.abs(ana)+np.abs(num)+1e-12)
print("解析:", ana); print("数值:", num); print("相对误差:", rel)
assert rel.max() < 1e-4, "梯度校验失败！"
print("梯度校验通过 ✓")

## 4 · 训练，打败 baseline

普通梯度下降，跑几百步，看 test accuracy 能不能超过 majority baseline（~0.6）。

In [ ]:
P = init(Xtr.shape[1], 16); lr = 0.5
for step in range(400):
    p, cache = forward(P, Xtr); g = backward(P, cache, ytr)
    for k in P: P[k] -= lr * g[k]
    if step % 100 == 0:
        acc = ((forward(P, Xte)[0] > 0.5) == yte).mean()
        print(f"step {step:3d}  train_loss={bce(p,ytr):.4f}  test_acc={acc:.3f}")
acc = ((forward(P, Xte)[0] > 0.5) == yte).mean()
print(f"\n最终 test_acc={acc:.3f}  vs baseline={baseline:.3f}  -> {'打败 ✓' if acc>baseline else '没打败 ✗'}")

## 5 · 从雅可比连乘看梯度消失（深层 sigmoid vs ReLU）

讲解第 7 节说：深层网络对浅层的梯度是一连串雅可比的连乘 $\nabla_{x_0}L=J_n^\top\cdots J_1^\top\nabla_{x_n}L$，每个 $\|J\|$ 略小于 1 就指数衰减（消失）。下面用真实乳腺癌特征当输入，搭一个 20 层的深堆叠，分别用 sigmoid（$\sigma'\le0.25$）和 ReLU（局部导数 0/1）激活，**实测**反向回传到第 1 层的梯度范数——你会看到 sigmoid 堆叠的浅层梯度被压扁了好几个数量级，而 ReLU 几乎不衰减。这就是「为什么深层网络要用 ReLU / 残差连接」的数值证据。

In [ ]:
# 把第 7 节的「雅可比连乘 -> 梯度消失」在真实数据上看个清楚。
# 搭一个 L 层的深堆叠，分别用 sigmoid 和 ReLU 激活，比较反向回传到第一层的梯度范数。
def deep_grad_norm(act, L=20, dh=30, seed=0):
    g = np.random.default_rng(seed)
    h = Xtr.copy()                       # 真实乳腺癌特征作输入 (n, 30)
    Ws, zs, hs = [], [], [h]
    for _ in range(L):
        W = g.normal(0, 1.0/np.sqrt(dh), (h.shape[1], dh))   # 同尺度初始化
        z = h @ W
        h = sigmoid(z) if act == "sigmoid" else np.maximum(0, z)
        Ws.append(W); zs.append(z); hs.append(h)
    # 反向：从最后一层一个任意上游梯度往回传，记录每层 dX 的范数
    dH = np.ones_like(h) / h.size
    norms = []
    for l in range(L-1, -1, -1):
        local = sigmoid(zs[l])*(1-sigmoid(zs[l])) if act == "sigmoid" else (zs[l] > 0).astype(float)
        dZ = dH * local
        dH = dZ @ Ws[l].T               # 传回上一层（乘 W^T = 雅可比转置）
        norms.append(np.linalg.norm(dH))
    return norms[::-1]                    # 第1层 -> 第L层

ns_sig = deep_grad_norm("sigmoid", L=20)
ns_relu = deep_grad_norm("relu", L=20)
print(f"sigmoid 堆叠: 第1层梯度范数={ns_sig[0]:.3e}  第20层={ns_sig[-1]:.3e}  衰减倍数={ns_sig[-1]/ns_sig[0]:.1e}x")
print(f"ReLU    堆叠: 第1层梯度范数={ns_relu[0]:.3e}  第20层={ns_relu[-1]:.3e}  衰减倍数={ns_relu[-1]/ns_relu[0]:.1e}x")
print("=> sigmoid 因 σ'≤0.25 连乘，回传到浅层的梯度被指数级压扁（梯度消失）；ReLU 局部导数为 0/1，衰减温和得多")

assert ns_sig[0] < ns_sig[-1] * 1e-3, "sigmoid 深堆叠应在浅层出现严重梯度消失"
assert ns_relu[0] > ns_sig[0], "同深度下 ReLU 的浅层梯度应明显大于 sigmoid"
print("梯度消失诊断验证通过 ✓  —— 这正是 ReLU/残差连接取代深层 sigmoid 的根因")

---
## ✏️ 练习区

`TODO` 骨架 + `assert` 自测。先自己写，全过再看文末答案。

### ✏️ 练习 1：sigmoid 与它的导数

实现 `sigmoid(z)`（数值稳定）和 `dsigmoid(z)`（即 $\sigma(z)(1-\sigma(z))$）。

In [ ]:
def sigmoid_safe(z):
    # TODO: 数值稳定的 sigmoid（提示：对 z>=0 和 z<0 分别处理避免 overflow）
    raise NotImplementedError

def dsigmoid(z):
    # TODO: sigmoid(z)*(1-sigmoid(z))
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
zt = np.array([-1000., -1., 0., 1., 1000.])
s = sigmoid_safe(zt)
assert np.isfinite(s).all() and abs(s[2]-0.5) < 1e-12
assert s[0] < 1e-50 and s[-1] > 1-1e-12
assert abs(dsigmoid(np.array([0.]))[0] - 0.25) < 1e-12   # 导数最大值
print("练习 1 通过 ✓")


### ✏️ 练习 2：线性层的反向（上游 × 局部）

前向 $Y = XW + b$。给定上游梯度 `dY`，实现 `linear_backward(X, W, dY)`，返回 `(dX, dW, db)`。
**用形状凑公式**：`dW` 必和 `W` 同形。

In [ ]:
def linear_backward(X, W, dY):
    # TODO: dX = dY @ W.T ; dW = X.T @ dY ; db = dY.sum(0)
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测：用数值梯度交叉验证 ——
g = np.random.default_rng(1)
X = g.normal(size=(6, 4)); W = g.normal(size=(4, 3)); b = g.normal(size=3)
dY = g.normal(size=(6, 3))
dX, dW, db = linear_backward(X, W, dY)
assert dX.shape==X.shape and dW.shape==W.shape and db.shape==b.shape
# 数值校验 dW
eps=1e-6; num=np.zeros_like(W)
for i in range(W.shape[0]):
  for j in range(W.shape[1]):
    Wp=W.copy();Wp[i,j]+=eps; Wm=W.copy();Wm[i,j]-=eps
    num[i,j]=((((X@Wp+b)*dY).sum())-(((X@Wm+b)*dY).sum()))/(2*eps)
assert np.allclose(dW, num, atol=1e-4)
print("练习 2 通过 ✓")


### ✏️ 练习 3：通用数值梯度校验器

实现 `numerical_grad(f, x, eps=1e-5)`：对标量函数 `f(x)` 用**中心差分**返回与 `x` 同形的梯度。
这是你以后调试任何 backprop 的工具。

In [ ]:
def numerical_grad(f, x, eps=1e-5):
    # TODO: 对 x 每个元素做中心差分；注意拷贝、别原地改 x
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测：在已知梯度的函数上验证 ——
f = lambda v: float((v**2).sum())          # grad = 2v
x = np.array([1., -2., 3.])
g_num = numerical_grad(f, x)
assert np.allclose(g_num, 2*x, atol=1e-4)
# 用它校验乳腺癌网络的 b2 梯度
P2 = init(Xtr.shape[1], 8)
ana_b2 = backward(P2, forward(P2, Xtr)[1], ytr)["b2"]
def lf(bv):
    Q={k:v.copy() for k,v in P2.items()}; Q["b2"]=bv; return loss_of(Q)
assert np.allclose(numerical_grad(lf, P2["b2"]), ana_b2, atol=1e-4)
print("练习 3 通过 ✓")


### ✏️ 练习 4：在乳腺癌数据上训练并达到 >95% test acc

用前面写好的 `forward`/`backward`，实现 `train_mlp(Xtr,ytr,Xte,yte,hidden,lr,steps)`，
返回最终 test accuracy。调超参，使其 **> 0.95**。

In [ ]:
def train_mlp(Xtr, ytr, Xte, yte, hidden=16, lr=0.5, steps=400):
    # TODO: init -> 循环 forward/backward/更新 -> 返回 test accuracy
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
acc = train_mlp(Xtr, ytr, Xte, yte, hidden=16, lr=0.5, steps=500)
print(f"test acc = {acc:.3f}")
assert acc > 0.95, "调大 steps 或 lr，标准化后乳腺癌很容易 >95%"
print("练习 4 通过 ✓")


---
## 📖 参考答案

In [ ]:
# 练习 1
def sigmoid_safe(z):
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1/(1+np.exp(-z[pos]))
    ez = np.exp(z[~pos]); out[~pos] = ez/(1+ez)
    return out
def dsigmoid(z):
    s = sigmoid_safe(z); return s*(1-s)
zt=np.array([-1000.,0.,1000.]); assert np.isfinite(sigmoid_safe(zt)).all()
assert abs(dsigmoid(np.array([0.]))[0]-0.25)<1e-12
print("练习 1 ✓")

In [ ]:
# 练习 2
def linear_backward(X, W, dY):
    return dY @ W.T, X.T @ dY, dY.sum(0)
print("练习 2 ✓")

In [ ]:
# 练习 3
def numerical_grad(f, x, eps=1e-5):
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=["multi_index"])
    while not it.finished:
        i = it.multi_index
        xp = x.copy(); xp[i]+=eps; xm = x.copy(); xm[i]-=eps
        g[i] = (f(xp)-f(xm))/(2*eps); it.iternext()
    return g
assert np.allclose(numerical_grad(lambda v:float((v**2).sum()), np.array([1.,-2.,3.])), [2,-4,6], atol=1e-4)
print("练习 3 ✓")

In [ ]:
# 练习 4
def train_mlp(Xtr, ytr, Xte, yte, hidden=16, lr=0.5, steps=400):
    P = init(Xtr.shape[1], hidden)
    for _ in range(steps):
        p, cache = forward(P, Xtr); g = backward(P, cache, ytr)
        for k in P: P[k] -= lr*g[k]
    return float(((forward(P, Xte)[0] > 0.5) == yte).mean())
assert train_mlp(Xtr,ytr,Xte,yte,16,0.5,500) > 0.95
print("练习 4 ✓ —— 你刚从零训练了一个网络诊断真实肿瘤数据")